## Test de Real ESRGAN — version Docker

Lancer depuis la racine du repo sur l'hôte :

```bash
./docker/build       # une seule fois (image real-esrgan:local)
./docker/jupyter     # JupyterLab sur http://localhost:8888, token affiché au démarrage
```

Le repo est bind-monté sur `/Real-ESRGAN` : tout ce qui est écrit dans `datasets/`,
`experiments/` et `results/` (tous gitignorés) reste sur l'hôte après l'arrêt du container.
Pas besoin de Google Drive.

Rien à installer : `basicsr`, `facexlib`, `gfpgan`, `requirements.txt` et l'install éditable
sont déjà dans l'image, et `docker/jupyter` relance `pip install -e .` au démarrage.
Le patch `torchvision.transforms.functional_tensor` est appliqué à la construction de l'image
(cf. `docker/README.md`), donc pas de `sed` à faire ici.

Les fichiers écrits depuis le container appartiennent à root sur l'hôte — la dernière cellule
du notebook remet les droits.

### Vérification de l'environnement

In [ ]:
import torch
from realesrgan import RealESRGANer  # échoue si l'install éditable n'a pas tourné
from realesrgan.version import __version__

print('realesrgan', __version__, '| torch', torch.__version__)
print('cuda disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device      :', torch.cuda.get_device_name(0))
    # la capability doit apparaître dans arch list, sinon chaque kernel meurt en
    # cudaErrorNoKernelImageForDevice (piège Blackwell, cf. docker/README.md)
    print('capability  : sm_%d%d' % torch.cuda.get_device_capability(0))
    print('arch list   :', ' '.join(torch.cuda.get_arch_list()))

In [ ]:
!nvidia-smi

### Chemins

In [ ]:
from pathlib import Path

REPO = Path('/Real-ESRGAN')            # repo bind-monté depuis l'hôte
WORK = REPO / 'datasets' / 'lea'       # données de travail, gitignoré -> persiste sur l'hôte
SPLIT = WORK / 'dataset_split'
TRAIN_DIR = SPLIT / 'train'
META_INFO = TRAIN_DIR / 'meta_info_train_pair.txt'
NAME = 'finetune_RealESRGANx2plus_pairdata_lea'
CONFIG = REPO / 'options' / 'finetune_realesrgan_x2plus_lea.yml'
PRETRAINED = REPO / 'experiments' / 'pretrained_models'

WORK.mkdir(parents=True, exist_ok=True)
PRETRAINED.mkdir(parents=True, exist_ok=True)
%cd /Real-ESRGAN
print('WORK =', WORK)

## Importation des données

Téléchargement depuis Google Drive dans `WORK`. Idempotent : ne retélécharge rien si les
dossiers sont déjà là. `unzip` n'existe pas dans l'image, on passe par `shutil`.

Si les données sont déjà sur l'hôte, il suffit de les placer dans
`datasets/lea/patches_in` et `datasets/lea/patches_out` et de sauter cette cellule.

In [ ]:
!pip install -q gdown

import shutil
import gdown

DRIVE_IDS = {
    'patches_out': '1BG0HqDYQZ0hJUwaLLs7LdVWe_liX3iZq',
    'patches_in': '14Z4jAx2YJA3W3pJSV71aHo4KGv-Bpzi_',
}

for name, file_id in DRIVE_IDS.items():
    if (WORK / name).exists():
        print(f'{name} déjà présent, saut')
        continue
    zip_path = WORK / f'{name}.zip'
    gdown.download(f'https://drive.google.com/uc?id={file_id}', str(zip_path), quiet=False)
    shutil.unpack_archive(zip_path, WORK)
    zip_path.unlink()

print(sorted(p.name for p in WORK.iterdir()))

### Filtrage des données avec nuage

In [ ]:
import glob
import os

import numpy as np
from PIL import Image

folder = str(WORK / 'patches_out')

mask_files = sorted(glob.glob(os.path.join(folder, '*_mask.png')))
print(f'Nombre de masques trouvés : {len(mask_files)}')

for f in mask_files[:5]:
    mask = np.array(Image.open(f))
    print(f'{os.path.basename(f)} -> valeurs uniques : {np.unique(mask)}')

all_values = set()
for f in mask_files:
    mask = np.array(Image.open(f))
    all_values.update(np.unique(mask).tolist())

print(f'\nToutes les valeurs uniques rencontrées dans tous les masques : {sorted(all_values)}')

In [ ]:
import glob
import os
import shutil

import numpy as np
from PIL import Image

folder_in = str(WORK / 'patches_in')
folder_out = str(WORK / 'patches_out')

folder_in_filtered = str(WORK / 'patches_in_filtered')
folder_out_filtered = str(WORK / 'patches_out_filtered')

os.makedirs(folder_in_filtered, exist_ok=True)
os.makedirs(folder_out_filtered, exist_ok=True)

CLOUD_VALUE = 128

mask_files = sorted(glob.glob(os.path.join(folder_out, '*_mask.png')))
print(f'Nombre de masques trouvés : {len(mask_files)}')

kept = 0
removed = 0

for mask_path in mask_files:
    mask = np.array(Image.open(mask_path))

    has_cloud = CLOUD_VALUE in np.unique(mask)

    mask_filename = os.path.basename(mask_path)
    base_name = mask_filename.replace('_mask.png', '')
    image_filename = base_name + '.png'

    if has_cloud:
        removed += 1
        continue

    kept += 1

    shutil.copy(os.path.join(folder_in, image_filename), os.path.join(folder_in_filtered, image_filename))
    shutil.copy(os.path.join(folder_in, mask_filename), os.path.join(folder_in_filtered, mask_filename))

    shutil.copy(os.path.join(folder_out, image_filename), os.path.join(folder_out_filtered, image_filename))
    shutil.copy(os.path.join(folder_out, mask_filename), os.path.join(folder_out_filtered, mask_filename))

print(f'\nPatches gardés (sans nuage) : {kept}')
print(f'Patches éliminés (avec nuage) : {removed}')
print(f'\nDossiers créés :')
print(f'  - {folder_in_filtered}')
print(f'  - {folder_out_filtered}')

### Split train / val / test

In [ ]:
import random

random.seed(42)

split_base = str(SPLIT)

image_files = sorted([
    f for f in os.listdir(folder_out_filtered) if f.endswith('.png') and not f.endswith('_mask.png')
])

random.shuffle(image_files)

n = len(image_files)
n_train = int(0.70 * n)
n_val = int(0.15 * n)
splits = {
    'train': image_files[:n_train],
    'val': image_files[n_train:n_train + n_val],
    'test': image_files[n_train + n_val:],
}

for split_name, files in splits.items():
    for sub in ['patches_in', 'patches_out']:
        os.makedirs(os.path.join(split_base, split_name, sub), exist_ok=True)

for split_name, files in splits.items():
    for image_filename in files:
        base_name = image_filename.replace('.png', '')
        mask_filename = base_name + '_mask.png'

        for sub, src_folder in [('patches_in', folder_in_filtered), ('patches_out', folder_out_filtered)]:
            dst_dir = os.path.join(split_base, split_name, sub)
            shutil.copy(os.path.join(src_folder, image_filename), os.path.join(dst_dir, image_filename))
            shutil.copy(os.path.join(src_folder, mask_filename), os.path.join(dst_dir, mask_filename))

print(f'Total patches : {n}')
print(f"Train : {len(splits['train'])} | Val : {len(splits['val'])} | Test : {len(splits['test'])}")

### Downscale des GT en 128x128 (entrées LQ du modèle x2)

In [ ]:
from PIL import Image

for split_name in ['train', 'val', 'test']:
    src_dir = os.path.join(split_base, split_name, 'patches_out')
    dst_dir = os.path.join(split_base, split_name, 'patches_out_128')
    os.makedirs(dst_dir, exist_ok=True)

    count = 0
    for fname in os.listdir(src_dir):
        if fname.endswith('.png') and not fname.endswith('_mask.png'):
            img = Image.open(os.path.join(src_dir, fname)).convert('RGB')
            img_resized = img.resize((128, 128), Image.BICUBIC)
            img_resized.save(os.path.join(dst_dir, fname))
            count += 1

    print(f'{split_name} : {count} images downscalées -> {dst_dir}')

In [ ]:
src = os.path.join(split_base, 'train', 'patches_in')
dst = os.path.join(split_base, 'train', 'patches_in_images')
os.makedirs(dst, exist_ok=True)

count = 0
for fname in os.listdir(src):
    if fname.endswith('.png') and not fname.endswith('_mask.png'):
        shutil.copy(os.path.join(src, fname), os.path.join(dst, fname))
        count += 1

print(f'{count} images copiées (sans masques) -> {dst}')

### Génération du meta_info

`--root` est explicite : sans lui le script calcule les chemins relatifs au répertoire courant
(`/Real-ESRGAN`), ce qui donne des `../...` que `dataroot_gt` ne sait pas résoudre.

In [ ]:
!python scripts/generate_meta_info_pairdata.py \
    --input {TRAIN_DIR}/patches_in_images {TRAIN_DIR}/patches_out_128 \
    --root {TRAIN_DIR} {TRAIN_DIR} \
    --meta_info {META_INFO}

In [ ]:
print(META_INFO.read_text().count('\n'), 'paires')
print(META_INFO.read_text().splitlines()[0])

## Téléchargement des modèles pré-entraînés

In [ ]:
BASE_URL = 'https://github.com/xinntao/Real-ESRGAN/releases/download'

# -nc : ne retélécharge pas si le fichier est déjà là
!wget -nc {BASE_URL}/v0.2.1/RealESRGAN_x2plus.pth -P {PRETRAINED}
!wget -nc {BASE_URL}/v0.2.2.3/RealESRGAN_x2plus_netD.pth -P {PRETRAINED}
!ls -lh {PRETRAINED}

## Fichier de configuration

scale = 2

rajout de `network_g: scale` sinon scale reste à 4

`dataroot_gt` et `dataroot_lq` pointent sur le split train, `meta_info` sur le fichier généré plus haut

`use_rot = True`

`total_iter` : 5000 pour commencer car pas beaucoup d'images, suffisant pour un premier test

`save_checkpoint_freq` = 500, pour rester cohérent avec le nombre d'itérations qui a été diminué

Écrit via f-string plutôt que `%%writefile` pour que les chemins restent ceux définis
dans la cellule « Chemins » — une seule source de vérité.

In [ ]:
CONFIG.write_text(f'''# general settings
name: {NAME}
model_type: RealESRGANModel
scale: 2
num_gpu: auto
manual_seed: 0

l1_gt_usm: True
percep_gt_usm: True
gan_gt_usm: False

high_order_degradation: False

datasets:
  train:
    name: AdjacencyLake
    type: RealESRGANPairedDataset
    dataroot_gt: {TRAIN_DIR}
    dataroot_lq: {TRAIN_DIR}
    meta_info: {META_INFO}
    io_backend:
      type: disk

    gt_size: 256
    use_hflip: True
    use_rot: True

    use_shuffle: true
    num_worker_per_gpu: 5
    batch_size_per_gpu: 4
    dataset_enlarge_ratio: 1
    prefetch_mode: ~

network_g:
  type: RRDBNet
  num_in_ch: 3
  num_out_ch: 3
  num_feat: 64
  num_block: 23
  num_grow_ch: 32
  scale: 2

network_d:
  type: UNetDiscriminatorSN
  num_in_ch: 3
  num_feat: 64
  skip_connection: True

path:
  pretrain_network_g: {PRETRAINED}/RealESRGAN_x2plus.pth
  param_key_g: params_ema
  strict_load_g: true
  pretrain_network_d: {PRETRAINED}/RealESRGAN_x2plus_netD.pth
  param_key_d: params
  strict_load_d: true
  resume_state: ~

train:
  ema_decay: 0.999
  optim_g:
    type: Adam
    lr: !!float 1e-4
    weight_decay: 0
    betas: [0.9, 0.99]
  optim_d:
    type: Adam
    lr: !!float 1e-4
    weight_decay: 0
    betas: [0.9, 0.99]

  scheduler:
    type: MultiStepLR
    milestones: [400000]
    gamma: 0.5

  total_iter: 5000
  warmup_iter: -1

  pixel_opt:
    type: L1Loss
    loss_weight: 1.0
    reduction: mean
  perceptual_opt:
    type: PerceptualLoss
    layer_weights:
      'conv1_2': 0.1
      'conv2_2': 0.1
      'conv3_4': 1
      'conv4_4': 1
      'conv5_4': 1
    vgg_type: vgg19
    use_input_norm: true
    perceptual_weight: !!float 1.0
    style_weight: 0
    range_norm: false
    criterion: l1
  gan_opt:
    type: GANLoss
    gan_type: vanilla
    real_label_val: 1.0
    fake_label_val: 0.0
    loss_weight: !!float 1e-1

  net_d_iters: 1
  net_d_init_iters: 0

logger:
  print_freq: 100
  save_checkpoint_freq: !!float 500
  use_tb_logger: true
  wandb:
    project: ~
    resume_id: ~

dist_params:
  backend: nccl
  port: 29500
''')

print(CONFIG.read_text())

## Entraînement

Les checkpoints vont dans `experiments/{NAME}/models/` — sur le bind-mount, donc directement
sur l'hôte. Aucune copie vers Drive nécessaire. `--auto_resume` reprend le dernier état si
le container est redémarré.

In [ ]:
%cd /Real-ESRGAN
!python realesrgan/train.py -opt {CONFIG} --auto_resume

## Test

### Downscale du test set en 128x128

In [ ]:
src_dir = os.path.join(split_base, 'test', 'patches_out')
dst_dir = os.path.join(split_base, 'test', 'patches_out_128')
os.makedirs(dst_dir, exist_ok=True)

count = 0
for fname in os.listdir(src_dir):
    if fname.endswith('.png') and not fname.endswith('_mask.png'):
        img = Image.open(os.path.join(src_dir, fname)).convert('RGB')
        img_resized = img.resize((128, 128), Image.BICUBIC)
        img_resized.save(os.path.join(dst_dir, fname))
        count += 1

print(f'test : {count} images downscalées -> {dst_dir}')

### Inférence

In [ ]:
CKPT = REPO / 'experiments' / NAME / 'models' / 'net_g_5000.pth'
assert CKPT.exists(), f'checkpoint absent : {CKPT}\n' + '\n'.join(
    str(p) for p in sorted((REPO / 'experiments' / NAME / 'models').glob('net_g_*.pth')))
print(CKPT)

In [ ]:
import numpy as np
from basicsr.archs.rrdbnet_arch import RRDBNet
from PIL import Image

from realesrgan import RealESRGANer

# Chargement du modèle fine-tuné
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)

upsampler = RealESRGANer(
    scale=2,
    model_path=str(CKPT),
    model=model,
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=True  # utilise la précision FP16, plus rapide sur GPU
)

test_lq_dir = os.path.join(split_base, 'test', 'patches_out_128')
test_gt_dir = os.path.join(split_base, 'test', 'patches_in')
output_dir = os.path.join(split_base, 'test', 'patches_output_inference')
os.makedirs(output_dir, exist_ok=True)


def calculate_psnr(img1, img2):
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    mse = np.mean((img1 - img2)**2)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(255.0 / np.sqrt(mse))


psnr_values = []
files = sorted([f for f in os.listdir(test_lq_dir) if f.endswith('.png')])

print(f'Nombre d\'images à traiter : {len(files)}')

for i, fname in enumerate(files):
    lq_path = os.path.join(test_lq_dir, fname)
    gt_path = os.path.join(test_gt_dir, fname)

    img_lq = np.array(Image.open(lq_path).convert('RGB'))

    output, _ = upsampler.enhance(img_lq, outscale=2)

    Image.fromarray(output).save(os.path.join(output_dir, fname))

    img_gt = np.array(Image.open(gt_path).convert('RGB'))

    psnr = calculate_psnr(output, img_gt)
    psnr_values.append(psnr)

    if (i + 1) % 50 == 0:
        print(f'{i+1}/{len(files)} images traitées, PSNR moyen jusqu\'ici : {np.mean(psnr_values):.2f} dB')

print(f'\n=== RÉSULTAT FINAL ===')
print(f'PSNR moyen sur le test set ({len(psnr_values)} images) : {np.mean(psnr_values):.2f} dB')
print(f'PSNR min : {np.min(psnr_values):.2f} dB')
print(f'PSNR max : {np.max(psnr_values):.2f} dB')

### Droits des fichiers

Le container tourne en root : tout ce qu'il écrit dans le bind-mount appartient à root sur
l'hôte. À lancer en fin de session pour rendre les fichiers à l'utilisateur hôte.

In [ ]:
!chown -R --reference=/Real-ESRGAN/setup.py /Real-ESRGAN/datasets /Real-ESRGAN/experiments 2>/dev/null || true
!ls -ld /Real-ESRGAN/datasets/lea /Real-ESRGAN/experiments/{NAME}